# UTS Intelegensi Buatan — Kasus 1: Klasifikasi Kultivar Anggur (`kasus01_wine.csv`)

- **Nama:** Hafizh Al Ma'arif Karim Amrullah
- **NIM:** `24552011137` (Jumlah digit: `2+4+5+5+2+0+1+1+1+3+7 = 31`, `31 mod 10 = 1` → **Kasus 1**)
- **Kelas:** TIF RP 23D
- **Mata Kuliah:** Intelegensi Buatan (3 SKS) — Universitas Teknologi Bandung
- **Dosen:** Muhammad Ikhwan Fathulloh, S.Kom.
- **Sumber Materi & Dataset:** [Artificial-Intelegent-Course-Bank](https://github.com/Muhammad-Ikhwan-Fathulloh/Artificial-Intelegent-Course-Bank)

Notebook ini menjalankan seluruh tahapan wajib UTS:
1. Instalasi dependensi & pengunduhan dataset `kasus01_wine.csv`
2. Audit data, analisis skala fitur, dan heatmap korelasi
3. Pembuatan modul `fitur.py` (5 fitur rasio + uji keputusan drop/keep fitur redundan)
4. Pembuatan dan eksekusi `latih.py` (split sebelum preprocessing, 5-Fold Stratified CV di data train, evaluasi test set 1x, simpan `models/kasus01_wine_pipeline.joblib`)
5. Pembuatan `app.py` dan peluncuran aplikasi **Streamlit** langsung dari Google Colab


## 1. Instalasi Dependensi & Unduh Dataset


In [ ]:
!pip -q install streamlit scikit-learn joblib pandas numpy matplotlib seaborn

from pathlib import Path
import urllib.request

Path("dataset").mkdir(exist_ok=True)
Path("models").mkdir(exist_ok=True)

DATA_URL = (
    "https://raw.githubusercontent.com/Muhammad-Ikhwan-Fathulloh/"
    "Artificial-Intelegent-Course-Bank/main/UTS/dataset/kasus01_wine.csv"
)
DATA_FILE = Path("dataset/kasus01_wine.csv")

if not DATA_FILE.exists():
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)
    print(f"Dataset berhasil diunduh ke {DATA_FILE}")
else:
    print(f"Dataset sudah tersedia di {DATA_FILE}")


## 2. Eksplorasi, Audit Data, Skala Fitur & Korelasi Antarfitur


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("dataset/kasus01_wine.csv")
print("Dimensi dataset:", df.shape)
print("Total missing value:", df.isna().sum().sum())
print("Total duplikat:", df.duplicated().sum())
print("\nDistribusi kelas target (kultivar):")
print(df["kultivar"].value_counts())

fitur_num = [c for c in df.columns if c != "kultivar"]
display(df[fitur_num].describe().T[["min", "50%", "mean", "max", "std"]].round(3))

# Visualisasi Heatmap Korelasi Antarfitur
plt.figure(figsize=(11, 8))
sns.heatmap(df[fitur_num].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Heatmap Korelasi 13 Komponen Kimia Anggur (kasus01_wine.csv)")
plt.tight_layout()
plt.show()


## 3. Tulis Modul `fitur.py` (Feature Engineering)


In [ ]:
%%writefile fitur.py
"""Modul Feature Engineering untuk Kasus 1: Klasifikasi Kultivar Anggur.

Nama  : Hafizh Al Ma'arif Karim Amrullah
NIM   : 24552011137 (Jumlah digit = 31, 31 mod 10 = 1 -> Kasus 1)
Kelas : TIF RP 23D
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

FITUR_MENTAH: list[str] = [
    "alcohol",
    "malic_acid",
    "ash",
    "alcalinity_of_ash",
    "magnesium",
    "total_phenols",
    "flavanoids",
    "nonflavanoid_phenols",
    "proanthocyanins",
    "color_intensity",
    "hue",
    "od280_od315_of_diluted_wines",
    "proline",
]

FITUR_RASIO: list[str] = [
    "rasio_flavanoid_fenol",
    "rasio_fenol_nonfenol",
    "rasio_warna_hue",
    "rasio_od_fenol",
    "rasio_prolin_alkohol",
]

KOLOM_TARGET: str = "kultivar"
EPS: float = 1e-6


def audit_data(df: pd.DataFrame, target_col: str = KOLOM_TARGET) -> dict:
    """Melakukan audit menyeluruh: dimensi, duplikat, missing value, tipe data, dan outlier IQR."""
    fitur_cols = [c for c in df.columns if c != target_col]
    outlier_per_kolom: dict[str, int] = {}

    for col in fitur_cols:
        if pd.api.types.is_numeric_dtype(df[col]):
            q1 = df[col].quantile(0.25)
            q3 = df[col].quantile(0.75)
            iqr = q3 - q1
            batas_bawah = q1 - 1.5 * iqr
            batas_atas = q3 + 1.5 * iqr
            outlier_per_kolom[col] = int(
                ((df[col] < batas_bawah) | (df[col] > batas_atas)).sum()
            )

    ringkasan = {
        "jumlah_baris": int(df.shape[0]),
        "jumlah_kolom": int(df.shape[1]),
        "jumlah_duplikat": int(df.duplicated().sum()),
        "missing_per_kolom": {col: int(val) for col, val in df.isna().sum().items()},
        "total_missing": int(df.isna().sum().sum()),
        "tipe_data": {col: str(dtype) for col, dtype in df.dtypes.items()},
        "distribusi_target": (
            df[target_col].value_counts().to_dict() if target_col in df.columns else {}
        ),
        "outlier_iqr": outlier_per_kolom,
    }
    return ringkasan


def analisis_korelasi_tinggi(
    X_train: pd.DataFrame, threshold: float = 0.60
) -> list[dict]:
    """Mengidentifikasi pasangan fitur dengan |r| >= threshold pada data train saja."""
    corr = X_train[FITUR_MENTAH].corr()
    pasangan: list[dict] = []
    cols = FITUR_MENTAH

    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            r_val = float(corr.iloc[i, j])
            if abs(r_val) >= threshold:
                pasangan.append(
                    {
                        "fitur_1": cols[i],
                        "fitur_2": cols[j],
                        "korelasi": round(r_val, 4),
                    }
                )

    pasangan.sort(key=lambda item: abs(item["korelasi"]), reverse=True)
    return pasangan


def buat_fitur_rasio(df: pd.DataFrame, drop_redundant: bool = False) -> pd.DataFrame:
    """Membuat 5 fitur rasio dari pasangan fitur kimia yang saling terkait.

    Pasangan fitur rasio:
    1. rasio_flavanoid_fenol : flavanoids / total_phenols
       Proporsi senyawa flavonoid terhadap total fenol.
    2. rasio_fenol_nonfenol  : flavanoids / nonflavanoid_phenols
       Keseimbangan senyawa fenolik flavonoid vs non-flavonoid.
    3. rasio_warna_hue       : color_intensity / hue
       Indeks kepekatan warna relatif terhadap rona warna anggur.
    4. rasio_od_fenol        : od280_od315_of_diluted_wines / total_phenols
       Rasio absorbansi UV terhadap total kadar fenol.
    5. rasio_prolin_alkohol  : proline / alcohol
       Keseimbangan asam amino prolin terhadap kadar alkohol.

    Parameter
    ---------
    drop_redundant : bool, default=False
        Jika True, membuang kolom 'total_phenols' yang berkorelasi sangat tinggi
        dengan 'flavanoids' dan sudah terwakili dalam fitur rasio.
    """
    hasil = df.copy()
    hasil = hasil[FITUR_MENTAH].astype(float)

    hasil["rasio_flavanoid_fenol"] = hasil["flavanoids"] / (hasil["total_phenols"] + EPS)
    hasil["rasio_fenol_nonfenol"] = hasil["flavanoids"] / (
        hasil["nonflavanoid_phenols"] + EPS
    )
    hasil["rasio_warna_hue"] = hasil["color_intensity"] / (hasil["hue"] + EPS)
    hasil["rasio_od_fenol"] = hasil["od280_od315_of_diluted_wines"] / (
        hasil["total_phenols"] + EPS
    )
    hasil["rasio_prolin_alkohol"] = hasil["proline"] / (hasil["alcohol"] + EPS)

    if drop_redundant:
        hasil = hasil.drop(columns=["total_phenols"])

    return hasil


class WineFeatureEngineer(BaseEstimator, TransformerMixin):
    """Custom transformer scikit-learn agar feature engineering berjalan di dalam Pipeline.

    Keuntungan:
    - Menerima DataFrame mentah (13 kolom asli) secara langsung saat training maupun inferensi.
    - Bebas data leakage karena seluruh transformasi bersifat baris-per-baris sebelum scaling.
    """

    def __init__(self, drop_redundant: bool = False):
        self.drop_redundant = drop_redundant

    def fit(self, X: pd.DataFrame, y=None):
        self.fitur_masuk_ = list(FITUR_MENTAH)
        contoh = buat_fitur_rasio(
            pd.DataFrame(X, columns=FITUR_MENTAH).iloc[:1],
            drop_redundant=self.drop_redundant,
        )
        self.fitur_keluar_ = list(contoh.columns)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if not isinstance(X, pd.DataFrame):
            X_df = pd.DataFrame(X, columns=FITUR_MENTAH)
        else:
            X_df = X
        return buat_fitur_rasio(X_df, drop_redundant=self.drop_redundant)

    def get_feature_names_out(self, input_features=None):
        return np.array(self.fitur_keluar_)


## 4. Tulis Skrip `latih.py` & Jalankan Pelatihan + Evaluasi


In [ ]:
%%writefile latih.py
"""Skrip Pelatihan, Validasi Silang, Evaluasi, dan Penyimpanan Model - Kasus 1 (Wine).

Nama  : Hafizh Al Ma'arif Karim Amrullah
NIM   : 24552011137 (Jumlah digit = 31, 31 mod 10 = 1 -> Kasus 1)
Kelas : TIF RP 23D

Aturan yang dipatuhi:
1. Split train/test dilakukan SEBELUM imputasi, scaling, atau seleksi fitur.
2. Pemilihan model & keputusan drop/keep fitur dilakukan dengan 5-Fold Stratified CV
   pada data train saja.
3. Evaluasi pada data test dilakukan tepat SATU KALI untuk model terbaik.
4. Pipeline lengkap + metadata disimpan ke models/kasus01_wine_pipeline.joblib tanpa path absolut.
"""

from __future__ import annotations

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from fitur import (
    FITUR_MENTAH,
    FITUR_RASIO,
    KOLOM_TARGET,
    WineFeatureEngineer,
    analisis_korelasi_tinggi,
    audit_data,
)

BASE_DIR = Path(__file__).resolve().parent
DATA_PATH = BASE_DIR / "dataset" / "kasus01_wine.csv"
MODELS_DIR = BASE_DIR / "models"
MODEL_PATH = MODELS_DIR / "kasus01_wine_pipeline.joblib"
SUMMARY_PATH = MODELS_DIR / "ringkasan_evaluasi.json"

GITHUB_RAW_URL = (
    "https://raw.githubusercontent.com/Muhammad-Ikhwan-Fathulloh/"
    "Artificial-Intelegent-Course-Bank/main/UTS/dataset/kasus01_wine.csv"
)


def muat_dataset() -> pd.DataFrame:
    """Memuat dataset lokal dari dataset/kasus01_wine.csv (atau mengunduh jika belum ada)."""
    if DATA_PATH.exists():
        return pd.read_csv(DATA_PATH)
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(GITHUB_RAW_URL)
    df.to_csv(DATA_PATH, index=False)
    return df


def buat_kandidat_pipeline() -> dict[str, Pipeline]:
    """Menyusun kandidat pipeline klasifikasi dengan imputasi dan StandardScaler."""
    return {
        "Logistic Regression (Keep + Rasio)": Pipeline(
            [
                ("fe", WineFeatureEngineer(drop_redundant=False)),
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=1000, C=1.0, random_state=42)),
            ]
        ),
        "Logistic Regression (Drop Redundan + Rasio)": Pipeline(
            [
                ("fe", WineFeatureEngineer(drop_redundant=True)),
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=1000, C=1.0, random_state=42)),
            ]
        ),
        "Random Forest (Keep + Rasio)": Pipeline(
            [
                ("fe", WineFeatureEngineer(drop_redundant=False)),
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                (
                    "model",
                    RandomForestClassifier(
                        n_estimators=200,
                        max_depth=None,
                        min_samples_split=2,
                        random_state=42,
                    ),
                ),
            ]
        ),
        "SVM RBF (Keep + Rasio)": Pipeline(
            [
                ("fe", WineFeatureEngineer(drop_redundant=False)),
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", SVC(kernel="rbf", C=1.0, probability=True, random_state=42)),
            ]
        ),
    }


def hitung_statistik_input(X_train: pd.DataFrame, y_train: pd.Series) -> tuple[dict, dict]:
    """Menghitung statistik input & contoh sampel per kultivar dari X_train saja."""
    statistik: dict[str, dict[str, float]] = {}
    for col in FITUR_MENTAH:
        statistik[col] = {
            "min": round(float(X_train[col].min()), 3),
            "median": round(float(X_train[col].median()), 3),
            "mean": round(float(X_train[col].mean()), 3),
            "max": round(float(X_train[col].max()), 3),
            "std": round(float(X_train[col].std()), 3),
        }

    contoh_sampel: dict[str, dict[str, float]] = {}
    for kelas in sorted(y_train.unique().tolist()):
        baris_kelas = X_train.loc[y_train == kelas].median()
        contoh_sampel[kelas] = {
            col: round(float(baris_kelas[col]), 3) for col in FITUR_MENTAH
        }

    return statistik, contoh_sampel


def main() -> None:
    print("=" * 72)
    print("PELATIHAN MODEL KLASIFIKASI - KASUS 1: KULTIVAR ANGGUR (kasus01_wine.csv)")
    print("=" * 72)

    # 1. Muat & audit dataset
    df = muat_dataset()
    hasil_audit = audit_data(df, target_col=KOLOM_TARGET)
    print(f"\n[1] Audit Dataset:")
    print(f"    - Dimensi         : {hasil_audit['jumlah_baris']} baris x {hasil_audit['jumlah_kolom']} kolom")
    print(f"    - Duplikat        : {hasil_audit['jumlah_duplikat']}")
    print(f"    - Total Missing   : {hasil_audit['total_missing']}")
    print(f"    - Distribusi Kelas: {hasil_audit['distribusi_target']}")

    # 2. Split train dan test SEBELUM preprocessing (anti data leakage)
    X = df[FITUR_MENTAH].copy()
    y = df[KOLOM_TARGET].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
    print(f"\n[2] Pembagian Data (Stratified 80/20 sebelum preprocessing):")
    print(f"    - Train: {len(X_train)} baris | Test: {len(X_test)} baris")

    # 3. Analisis korelasi antarfitur pada data train saja
    korelasi_tinggi = analisis_korelasi_tinggi(X_train, threshold=0.60)
    print("\n[3] Pasangan Fitur Berkorelasi Tinggi pada Data Train (|r| >= 0.60):")
    for item in korelasi_tinggi:
        print(f"    - {item['fitur_1']:30s} <-> {item['fitur_2']:30s}: r = {item['korelasi']:+.4f}")

    # 4. Perbandingan model menggunakan 5-Fold Stratified Cross-Validation di data train
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    kandidat = buat_kandidat_pipeline()
    perbandingan_model: list[dict] = []

    print("\n[4] Perbandingan Model (5-Fold Stratified CV pada Data Train):")
    for nama, pipe in kandidat.items():
        scores = cross_validate(
            pipe,
            X_train,
            y_train,
            cv=cv,
            scoring=["accuracy", "f1_macro", "f1_weighted"],
            n_jobs=-1,
        )
        ringkas = {
            "model": nama,
            "cv_accuracy_mean": round(float(scores["test_accuracy"].mean()), 4),
            "cv_accuracy_std": round(float(scores["test_accuracy"].std()), 4),
            "cv_f1_macro_mean": round(float(scores["test_f1_macro"].mean()), 4),
            "cv_f1_macro_std": round(float(scores["test_f1_macro"].std()), 4),
            "cv_f1_weighted_mean": round(float(scores["test_f1_weighted"].mean()), 4),
        }
        perbandingan_model.append(ringkas)
        print(
            f"    - {nama:44s} | Acc: {ringkas['cv_accuracy_mean']:.4f} (+/- {ringkas['cv_accuracy_std']:.4f}) "
            f"| F1-Macro: {ringkas['cv_f1_macro_mean']:.4f} (+/- {ringkas['cv_f1_macro_std']:.4f})"
        )

    # Pilih model terbaik berdasarkan CV F1-Macro, lalu CV Accuracy, lalu std terkecil
    perbandingan_model.sort(
        key=lambda r: (r["cv_f1_macro_mean"], r["cv_accuracy_mean"], -r["cv_f1_macro_std"]),
        reverse=True,
    )
    terbaik_info = perbandingan_model[0]
    nama_terbaik = terbaik_info["model"]
    pipeline_terbaik = kandidat[nama_terbaik]
    print(f"\n    ==> Model Terbaik (berdasarkan CV Data Train): {nama_terbaik}")

    # 5. Latih ulang model terbaik pada seluruh data train, evaluasi 1 KALI di data test
    pipeline_terbaik.fit(X_train, y_train)
    y_pred = pipeline_terbaik.predict(X_test)
    y_proba = pipeline_terbaik.predict_proba(X_test)
    kelas = list(pipeline_terbaik.classes_)

    metrik_test = {
        "accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
        "precision_macro": round(float(precision_score(y_test, y_pred, average="macro")), 4),
        "recall_macro": round(float(recall_score(y_test, y_pred, average="macro")), 4),
        "f1_macro": round(float(f1_score(y_test, y_pred, average="macro")), 4),
        "f1_weighted": round(float(f1_score(y_test, y_pred, average="weighted")), 4),
        "roc_auc_ovr_macro": round(
            float(roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")), 4
        ),
    }
    cm = confusion_matrix(y_test, y_pred, labels=kelas)
    cls_report_dict = classification_report(
        y_test, y_pred, target_names=kelas, output_dict=True
    )

    print("\n[5] Evaluasi Final pada Data Test (Dieksekusi Tepat 1 Kali):")
    for k, v in metrik_test.items():
        print(f"    - {k:20s}: {v:.4f}")
    print("\n    Classification Report:")
    print(classification_report(y_test, y_pred, target_names=kelas))
    print("    Confusion Matrix (baris=Aktual, kolom=Prediksi):")
    print(pd.DataFrame(cm, index=kelas, columns=kelas))

    # 6. Simpan pipeline lengkap + metadata dengan joblib
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    statistik_input, contoh_sampel = hitung_statistik_input(X_train, y_train)

    bundle = {
        "pipeline": pipeline_terbaik,
        "fitur": FITUR_MENTAH,
        "fitur_rasio": FITUR_RASIO,
        "kelas": kelas,
        "nama_model_terbaik": nama_terbaik,
        "statistik_input": statistik_input,
        "contoh_sampel": contoh_sampel,
        "audit_data": hasil_audit,
        "korelasi_tinggi": korelasi_tinggi,
        "perbandingan_model": perbandingan_model,
        "metrik_cv": terbaik_info,
        "metrik_test": metrik_test,
        "confusion_matrix": cm.tolist(),
        "classification_report": cls_report_dict,
        "versi_sklearn": sklearn.__version__,
    }

    joblib.dump(bundle, MODEL_PATH, compress=3)
    print(f"\n[6] Model & metadata berhasil disimpan ke: {MODEL_PATH.relative_to(BASE_DIR)}")

    # Simpan ringkasan JSON (tanpa objek pipeline) untuk keperluan pelaporan
    ringkasan_json = {k: v for k, v in bundle.items() if k != "pipeline"}
    SUMMARY_PATH.write_text(
        json.dumps(ringkasan_json, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print(f"    Ringkasan evaluasi JSON disimpan ke : {SUMMARY_PATH.relative_to(BASE_DIR)}")


if __name__ == "__main__":
    main()


In [ ]:
!python latih.py


## 5. Visualisasi Confusion Matrix & Perbandingan Model (dari Metadata `joblib`)


In [ ]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import fitur  # noqa: F401

bundle = joblib.load("models/kasus01_wine_pipeline.joblib")
print("Model Terbaik:", bundle["nama_model_terbaik"])
print("Metrik Test Set:", bundle["metrik_test"])

display(pd.DataFrame(bundle["perbandingan_model"]))

cm = bundle["confusion_matrix"]
kelas = bundle["kelas"]
plt.figure(figsize=(6, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=kelas, yticklabels=kelas)
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.title(f"Confusion Matrix Test Set — {bundle['nama_model_terbaik']}")
plt.tight_layout()
plt.show()


## 6. Tulis Aplikasi Streamlit (`app.py`)


In [ ]:
%%writefile app.py
"""Aplikasi Streamlit Klasifikasi Kultivar Anggur - UTS Intelegensi Buatan (Kasus 1).

Nama  : Hafizh Al Ma'arif Karim Amrullah
NIM   : 24552011137 (Kasus 1: kasus01_wine.csv)
Kelas : TIF RP 23D
"""

from __future__ import annotations

from pathlib import Path

import joblib
import pandas as pd
import streamlit as st

# Import modul fitur agar custom transformer WineFeatureEngineer dikenali saat joblib.load()
import fitur  # noqa: F401
from fitur import FITUR_MENTAH, buat_fitur_rasio

BASE_DIR = Path(__file__).resolve().parent
MODEL_PATH = BASE_DIR / "models" / "kasus01_wine_pipeline.joblib"

LABEL_FITUR: dict[str, str] = {
    "alcohol": "Kadar Alkohol (%)",
    "malic_acid": "Asam Malat (g/L)",
    "ash": "Kadar Abu (g/L)",
    "alcalinity_of_ash": "Alkalinitas Abu",
    "magnesium": "Magnesium (mg/L)",
    "total_phenols": "Total Fenol",
    "flavanoids": "Flavonoid",
    "nonflavanoid_phenols": "Fenol Non-Flavonoid",
    "proanthocyanins": "Proantosianin",
    "color_intensity": "Intensitas Warna",
    "hue": "Rona Warna (Hue)",
    "od280_od315_of_diluted_wines": "Rasio Absorbansi OD280/OD315",
    "proline": "Kadar Prolin (mg/L)",
}

st.set_page_config(
    page_title="Klasifikasi Kultivar Anggur — UTS AI (Kasus 1)",
    page_icon="🍷",
    layout="wide",
)


@st.cache_resource
def muat_bundle() -> dict:
    if not MODEL_PATH.exists():
        st.error(
            "File model `models/kasus01_wine_pipeline.joblib` belum ditemukan. "
            "Jalankan `python latih.py` terlebih dahulu."
        )
        st.stop()
    return joblib.load(MODEL_PATH)


bundle = muat_bundle()
pipeline = bundle["pipeline"]
kelas_list: list[str] = bundle["kelas"]
stat_input: dict = bundle["statistik_input"]
contoh_sampel: dict = bundle["contoh_sampel"]
metrik_test: dict = bundle["metrik_test"]
metrik_cv: dict = bundle["metrik_cv"]

st.title("Klasifikasi Kultivar Anggur dari Analisis Kimia")
st.caption(
    f"UTS Intelegensi Buatan — Kasus 1 (`kasus01_wine.csv`) | "
    f"Hafizh Al Ma'arif Karim Amrullah (NIM 24552011137, TIF RP 23D) | "
    f"Model aktif: **{bundle['nama_model_terbaik']}**"
)

tab_prediksi, tab_evaluasi, tab_fe = st.tabs(
    [
        "1. Prediksi Sampel (Input Mentah)",
        "2. Metrik Evaluasi & Perbandingan Model",
        "3. Feature Engineering & Metadata",
    ]
)

with tab_prediksi:
    st.subheader("Masukkan 13 Komponen Kimia Sampel Anggur")
    st.write(
        "Aplikasi menerima **input mentah** secara langsung. Transformasi fitur rasio, "
        "imputasi median, dan `StandardScaler` dijalankan otomatis di dalam pipeline."
    )

    # Preset contoh cepat untuk pengujian
    if "nilai_input" not in st.session_state:
        st.session_state["nilai_input"] = {
            col: float(stat_input[col]["median"]) for col in FITUR_MENTAH
        }

    preset_cols = st.columns(len(kelas_list) + 1)
    for idx, nama_kelas in enumerate(kelas_list):
        if preset_cols[idx].button(f"Contoh {nama_kelas}", use_container_width=True):
            st.session_state["nilai_input"] = {
                col: float(contoh_sampel[nama_kelas][col]) for col in FITUR_MENTAH
            }
    if preset_cols[-1].button("Reset ke Median", use_container_width=True):
        st.session_state["nilai_input"] = {
            col: float(stat_input[col]["median"]) for col in FITUR_MENTAH
        }

    col1, col2, col3 = st.columns(3)
    grup_1 = ["alcohol", "malic_acid", "ash", "alcalinity_of_ash", "magnesium"]
    grup_2 = ["total_phenols", "flavanoids", "nonflavanoid_phenols", "proanthocyanins"]
    grup_3 = ["color_intensity", "hue", "od280_od315_of_diluted_wines", "proline"]

    input_pengguna: dict[str, float] = {}

    with col1:
        st.markdown("**Komposisi Dasar & Mineral**")
        for col in grup_1:
            input_pengguna[col] = st.number_input(
                LABEL_FITUR[col],
                min_value=0.0,
                max_value=float(stat_input[col]["max"] * 2.0),
                value=float(st.session_state["nilai_input"][col]),
                step=0.1 if col != "magnesium" else 1.0,
                format="%.3f",
                key=f"in_{col}",
            )

    with col2:
        st.markdown("**Komponen Fenolik**")
        for col in grup_2:
            input_pengguna[col] = st.number_input(
                LABEL_FITUR[col],
                min_value=0.0,
                max_value=float(stat_input[col]["max"] * 2.0),
                value=float(st.session_state["nilai_input"][col]),
                step=0.05,
                format="%.3f",
                key=f"in_{col}",
            )

    with col3:
        st.markdown("**Warna, Absorbansi & Prolin**")
        for col in grup_3:
            input_pengguna[col] = st.number_input(
                LABEL_FITUR[col],
                min_value=0.0,
                max_value=float(stat_input[col]["max"] * 2.0),
                value=float(st.session_state["nilai_input"][col]),
                step=10.0 if col == "proline" else 0.05,
                format="%.3f",
                key=f"in_{col}",
            )

    if st.button("Prediksi Kultivar", type="primary", use_container_width=True):
        df_mentah = pd.DataFrame([input_pengguna])[bundle["fitur"]]
        prediksi = pipeline.predict(df_mentah)[0]
        probabilitas = pipeline.predict_proba(df_mentah)[0]
        prob_tertinggi = float(probabilitas.max())

        st.divider()
        res_kiri, res_kanan = st.columns([1, 1.3])

        with res_kiri:
            st.success(f"Hasil Prediksi: **{prediksi}**")
            st.metric("Probabilitas Kelas Terpilih", f"{prob_tertinggi * 100:.2f}%")
            df_prob = pd.DataFrame(
                {
                    "Kultivar": kelas_list,
                    "Probabilitas": [round(float(p), 4) for p in probabilitas],
                    "Persentase (%)": [f"{p * 100:.2f}%" for p in probabilitas],
                }
            )
            st.dataframe(df_prob, hide_index=True, use_container_width=True)

        with res_kanan:
            st.markdown("**Distribusi Probabilitas per Kultivar**")
            chart_df = pd.DataFrame(
                {"Probabilitas": probabilitas}, index=kelas_list
            )
            st.bar_chart(chart_df)

        with st.expander("Lihat 5 Fitur Rasio Turunan yang Dihitung Otomatis"):
            df_rasio = buat_fitur_rasio(df_mentah)[bundle["fitur_rasio"]]
            st.dataframe(df_rasio.round(4), hide_index=True, use_container_width=True)

with tab_evaluasi:
    st.subheader("Hasil Evaluasi Final pada Data Test (Dieksekusi 1 Kali)")
    m1, m2, m3, m4 = st.columns(4)
    m1.metric("Test Accuracy", f"{metrik_test['accuracy'] * 100:.2f}%")
    m2.metric("Test Macro F1", f"{metrik_test['f1_macro'] * 100:.2f}%")
    m3.metric("Test Weighted F1", f"{metrik_test['f1_weighted'] * 100:.2f}%")
    m4.metric("ROC-AUC (OvR Macro)", f"{metrik_test['roc_auc_ovr_macro'] * 100:.2f}%")

    st.subheader("Perbandingan Model (5-Fold Stratified CV pada Data Train)")
    df_perbandingan = pd.DataFrame(bundle["perbandingan_model"])
    st.dataframe(df_perbandingan, hide_index=True, use_container_width=True)

    c_kiri, c_kanan = st.columns(2)
    with c_kiri:
        st.subheader("Confusion Matrix (Data Test)")
        df_cm = pd.DataFrame(
            bundle["confusion_matrix"],
            index=[f"Aktual: {k}" for k in kelas_list],
            columns=[f"Pred: {k}" for k in kelas_list],
        )
        st.dataframe(df_cm, use_container_width=True)

    with c_kanan:
        st.subheader("Metrik per Kelas (Data Test)")
        rows_cls = []
        for k in kelas_list:
            info = bundle["classification_report"][k]
            rows_cls.append(
                {
                    "Kelas": k,
                    "Precision": round(info["precision"], 4),
                    "Recall": round(info["recall"], 4),
                    "F1-Score": round(info["f1-score"], 4),
                    "Support": int(info["support"]),
                }
            )
        st.dataframe(pd.DataFrame(rows_cls), hide_index=True, use_container_width=True)

with tab_fe:
    st.subheader("Pasangan Fitur Berkorelasi Tinggi (Data Train, |r| >= 0.60)")
    st.dataframe(
        pd.DataFrame(bundle["korelasi_tinggi"]),
        hide_index=True,
        use_container_width=True,
    )

    st.subheader("Metadata Pipeline Tersimpan")
    st.json(
        {
            "nama_model_terbaik": bundle["nama_model_terbaik"],
            "jumlah_fitur_mentah": len(bundle["fitur"]),
            "fitur_rasio_tambahan": bundle["fitur_rasio"],
            "kelas_target": bundle["kelas"],
            "cv_f1_macro_mean": metrik_cv["cv_f1_macro_mean"],
            "versi_sklearn": bundle["versi_sklearn"],
        }
    )


## 7. Jalankan Aplikasi Streamlit di Google Colab

Sel di bawah mengunduh `cloudflared`, menjalankan `streamlit run app.py` di port `8501`, lalu menampilkan URL publik (`https://*.trycloudflare.com`) yang dapat langsung dibuka di browser tanpa halaman password.


In [ ]:
import re
import subprocess
import time
import urllib.request
from pathlib import Path

# Unduh binary cloudflared jika belum tersedia
cf_path = Path("cloudflared")
if not cf_path.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cf_path,
    )
    cf_path.chmod(0o755)

# Jalankan Streamlit di background
st_log = open("streamlit.log", "w")
st_proc = subprocess.Popen(
    [
        "streamlit",
        "run",
        "app.py",
        "--server.port",
        "8501",
        "--server.headless",
        "true",
        "--server.enableCORS",
        "false",
    ],
    stdout=st_log,
    stderr=st_log,
)
time.sleep(3)

# Buka tunnel publik Cloudflare
cf_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for _ in range(60):
    line = cf_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print(f"Aplikasi Streamlit aktif di: {public_url}")
else:
    print("Gagal menangkap URL publik. Periksa streamlit.log untuk detail.")
